# Benchmark du retrieval

Mesure reproductible par sous-ensemble. Commencer par `regression`, puis `development`; conserver `test` pour la validation finale.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
cfg = load_config(ROOT / "config.yaml")
print(f"Projet : {ROOT}")


Projet : <PROJECT_ROOT>


In [2]:
import logging, pandas as pd
from evaluation import load_golden_set, RetrievalEvaluator, save_evaluation_run
from observability import new_run_id
from rag_pipeline import RAGPipeline

logging.basicConfig(level=logging.INFO)
pipeline = RAGPipeline.from_config(ROOT / "config.yaml")
all_examples = load_golden_set(cfg.evaluation.expanded_golden_set_full_path)
print(f"{len(all_examples)} cas disponibles")


INFO:FlagEmbedding.finetune.embedder.encoder_only.m3.runner:loading existing colbert_linear and sparse_linear---------
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


100 cas disponibles


In [3]:
RUN_SCOPE = "development" # puis "development"; ne lancer "test" qu'après choix des paramètres
examples = [item for item in all_examples if item.split == RUN_SCOPE]
print(f"Sous-ensemble : {RUN_SCOPE} — {len(examples)} questions")

# Échauffement GPU hors métriques
pipeline.retriever.retrieve("Quelles formations propose l'ENSEA ?")


Sous-ensemble : development — 40 questions


INFO:vectorstore:Cache RAM des vecteurs chargé : 4459 vecteurs (17.4 Mo).
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:retriever:Retrieval 13.747s | query=0.037s dense=0.034s bm25=0.003s fusion=0.000s vectors+MMR=1.111s rerank=12.562s | 5 résultat(s)


[RetrievedChunk(chunk_id='34878c2eb25043d3_f05e6b5e9a3d2ddb_a43c0dd6128646ae', url='https://ensea360.ensea.ed.ci/formation-continue/client-particulier/', title='Client particulier - ENSEA', section='Client particulier', doc_type='page', category='formation-continue', date='2025-10-30T12:36:59+00:00', text='# Client particulier  \nENSEA Formation Continue est l’unité de formation continue de L’École Nationale Supérieur de Statistique d’Économie Appliquée (ENSEA). Son objectif principal est de  \ndélivrer des formations exigeantes aux contenus scientifiques innovants aux entreprises publiques et privés. Ses formateurs de premier plan sont des experts dans leur domaine d’intervention.  \nGrâce à ses liens étroits avec le monde du travail et celui de la recherche, l’ENSEA Formation Continue crée une synergie forte entre formation continue, formation initiale et expériences des praticiens. Ces atouts essentiels permettent de proposer aux professionnels des formations de haut niveau.', paren

In [4]:
run_id = new_run_id(f"eval-{RUN_SCOPE}")
evaluator = RetrievalEvaluator(cfg.evaluation)
summary = evaluator.evaluate(pipeline, examples, run_id=run_id)
display(pd.Series(summary, name="valeur").to_frame())


INFO:retriever:Retrieval 21.505s | query=0.030s dense=0.019s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=21.442s | 5 résultat(s)
INFO:retriever:Retrieval 20.801s | query=0.029s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.010s rerank=20.741s | 3 résultat(s)
INFO:retriever:Retrieval 8.900s | query=0.029s dense=0.018s bm25=0.002s fusion=0.000s vectors+MMR=0.009s rerank=8.841s | 5 résultat(s)
INFO:retriever:Retrieval 24.378s | query=0.028s dense=0.016s bm25=0.004s fusion=0.000s vectors+MMR=0.009s rerank=24.321s | 5 résultat(s)
INFO:retriever:Retrieval 24.409s | query=0.031s dense=0.024s bm25=0.005s fusion=0.000s vectors+MMR=0.011s rerank=24.338s | 2 résultat(s)
INFO:retriever:Retrieval 8.979s | query=0.032s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.010s rerank=8.914s | 5 résultat(s)
INFO:retriever:Retrieval 4.988s | query=0.046s dense=0.036s bm25=0.011s fusion=0.000s vectors+MMR=0.012s rerank=4.882s | 5 résultat(s)
INFO:retriever:Retrieval 24.695s | query=0.039s

,valeur
questions,40.000000
answerable_questions,40.000000
unanswerable_questions,0.000000
mrr,0.812500
mean_latency_seconds,24.160990
p50_latency_seconds,23.690721
p95_latency_seconds,46.362593
max_latency_seconds,93.222233
recall@3,0.716667
precision@3,0.370833


In [5]:
for dimension, values in evaluator.last_group_metrics.items():
    print(f"\nRésultats par {dimension}")
    display(pd.DataFrame.from_dict(values, orient="index"))

details = pd.DataFrame(evaluator.last_details)
answerable = details[details["should_answer"]]
misses = answerable[answerable["first_relevant_rank"].isna()]
print(f"Échecs stricts : {len(misses)}/{len(answerable)}")
display(misses[["id", "question", "category", "top_title", "top_url", "relevant_urls"]])



Résultats par category


,questions,mrr,mean_latency_seconds,hit_rate@3,hit_rate@5,hit_rate@10
admission,4,0.833333,31.874845,1.00,1.00,1.00
contact,1,1.000000,23.690721,1.00,1.00,1.00
cooperation,5,0.700000,8.398389,0.80,0.80,0.80
doctorat,4,0.750000,17.698785,0.75,0.75,0.75
financement,4,0.500000,21.745624,0.50,0.50,0.50
formation_continue,2,0.666667,24.605120,1.00,1.00,1.00
formation_initiale,8,0.916667,33.119681,1.00,1.00,1.00
gouvernance,1,1.000000,23.767168,1.00,1.00,1.00
insertion,1,1.000000,19.223467,1.00,1.00,1.00
institution,1,1.000000,8.899701,1.00,1.00,1.00



Résultats par difficulty


,questions,mrr,mean_latency_seconds,hit_rate@3,hit_rate@5,hit_rate@10
easy,2,1.000000,22.604969,1.000000,1.000000,1.000000
hard,4,1.000000,44.226632,1.000000,1.000000,1.000000
medium,34,0.779412,21.891856,0.852941,0.852941,0.852941



Résultats par split


,questions,mrr,mean_latency_seconds,hit_rate@3,hit_rate@5,hit_rate@10
development,40,0.8125,24.16099,0.875,0.875,0.875


Échecs stricts : 5/40


,id,question,category,top_title,top_url,relevant_urls
13,ext-014,Où consulter la liste des doctorants de l'ENSEA ?,doctorat,Admission Doctorat - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/formation-doctorale/docto...
14,ext-015,L'ENSEA organise-t-elle des séminaires de rech...,recherche,765106d61b6f9473,https://ensea.ed.ci/wp-content/uploads/2024/06...,[https://ensea360.ensea.ed.ci/formation-contin...
15,ext-016,Quels organismes sont partenaires de recherche...,cooperation,Nos statistiques - ENSEA,https://ensea360.ensea.ed.ci/nos-statistiques/,[https://ensea.ed.ci/cooperation/partenaires-d...
26,ext-027,Où sont indiqués les coûts des formations init...,financement,,,[https://ensea.ed.ci/formation-initiale/couts/]
27,ext-028,Où puis-je consulter les tarifs de l'ENSEA ?,financement,Client entreprise - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/faq-items/quels-sont-les-...


In [6]:
run_dir = save_evaluation_run(
    cfg.evaluation.runs_full_path, summary, evaluator.last_details,
    evaluator.last_group_metrics, run_name=f"retrieval_{RUN_SCOPE}",
)
print(f"Résultats sauvegardés dans : {run_dir}")


Résultats sauvegardés dans : <PROJECT_ROOT>\data\evaluation_runs\20260803T135802Z_retrieval_development
